In [5]:
%load_ext line_profiler
%load_ext autoreload
%autoreload 2




The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:

import numpy as np
from box import Box
import numba

from datasetz.core.load_dataset import load_embedded_dataset
from mlflow import MlflowClient
from mlutils.mlflow.utils import get_run_params, terminate_run, finish_run_and_print_exception
from more_itertools import grouper
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import ElementwiseProblem
from pymoo.optimize import minimize
from sklearn.ensemble import RandomForestClassifier
from mlutils.pymoo.utils import JoblibParallelization
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import cross_validate, RepeatedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.tree import DecisionTreeClassifier
from toolz.curried import pipe

from rules.classification.competence_region_ensemble import SimpleCompetenceRegionEnsemble


In [7]:
def nn_wrapper(nn):
    return Box({
        "predict": lambda x: nn.kneighbors(x, n_neighbors=nn.n_samples_fit_, return_distance=False)
    })



In [4]:
def create_estimator(centroids, rf, x):
    
    space_classifier = NearestNeighbors()
    space_classifier.fit(centroids)
    wrapped_space_classifier = nn_wrapper(space_classifier)

    space_preds = wrapped_space_classifier.predict(x)

    best_tree_by_centroid = {}
    for idx, centroid in enumerate(centroids):
        x_in_centr = x[space_preds[:, 1] == idx]
        rf_preds = rf.predict(x_in_centr)
        for tree in rf.estimators_:
            tree_preds = tree.predict(x_in_centr)
            depth = tree.get_depth()
            score = 1/depth + accuracy_score(rf_preds, tree_preds)

            if not idx in best_tree_by_centroid.keys():
                best_tree_by_centroid[idx] = {
                    "tree": tree,
                    "score": score
                }
            elif score > best_tree_by_centroid[idx]["score"]:
                best_tree_by_centroid[idx] = {
                    "tree": tree,
                    "score": score
                }
        
        
    model = SimpleCompetenceRegionEnsemble(
        wrapped_space_classifier,
        {label: tree_details["tree"] for label, tree_details in best_tree_by_centroid.items()}
    )

    return model, space_classifier

In [25]:
def individual_to_centroid(indv, n_dim: int):
    return pipe(
        indv,
        lambda x: grouper(x, n_dim),
        list,
        np.array,
        np.nan_to_num
    )

In [26]:

class OptimalCentroidExplainer(ElementwiseProblem):
    def __init__(self, rf, n_clusters, x_train, y_train, *args, **kwargs):
        n_dim = x_train.shape[1]

        super().__init__(
            n_var=n_clusters * n_dim,  # each centroid * number of features 
            n_obj=2,
            n_constr=0,
            xl=list(np.min(x_train, axis=0)) * n_clusters,
            xu=list(np.max(x_train, axis=0)) * n_clusters,
            *args,
            **kwargs
        )

        self.n_clusters = n_clusters
        self.x_train = x_train
        self.y_train = y_train
        self.n_dim = n_dim
        self.rf = rf
        

    def build_model(self, individual):
        n_coordinates_in_individual = self.n_dim * self.n_clusters
        centroid_coordinates = individual[:n_coordinates_in_individual]

        individual_as_centroids = individual_to_centroid(centroid_coordinates, self.n_dim)
        
        return create_estimator(individual_as_centroids, self.rf, self.x_train)

    
    def _evaluate(self, individual, out, *args, **kwargs):
        n_coordinates_in_individual = self.n_dim * self.n_clusters
        centroid_coordinates = individual[:n_coordinates_in_individual]

        individual_as_centroids = individual_to_centroid(centroid_coordinates, self.n_dim)

        try:
            model, space_classifier = create_estimator(individual_as_centroids, self.rf, self.x_train)
        except Exception as e:
            print(e)
            out["F"] = [1, 9999]
            return
        skf = RepeatedKFold(n_splits=3, n_repeats=3, random_state=42)
        scores = cross_validate(model, self.x_train, self.rf.predict(self.x_train), n_jobs=1, scoring='accuracy',
                                cv=skf, fit_params={
                'competence_region_classifier': nn_wrapper(space_classifier)
            })
        acc = scores['test_score'].mean()
        mean_depth = np.average([tree.get_depth() for tree in model.clf_by_label.values()])
     
        print(f"Acc = {acc}, depth = {mean_depth}")

        out["F"] = [1 - acc, mean_depth]

In [28]:
def do_experiment(run_id):
    client = MlflowClient(tracking_uri="http://192.168.1.181:5010")
    params = get_run_params(run_id, client)

    from loguru import logger
    logger.info(params)

    try:
        dataset = load_embedded_dataset('keel-embedded', params.dataset_name)
        # DATA and preprocessing
        
        splitter = StratifiedShuffleSplit(random_state=42, n_splits=2)
        train_test_dataset = (dataset \
                              .encode_x_to_labels() \
                              .encode_y_to_numeric_labels() \
                              .train_test_split(splitter))[int(params.dataset_split)]
        dt = DecisionTreeClassifier(random_state=42)
        dt.fit(train_test_dataset.train.x, train_test_dataset.train.y)
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(train_test_dataset.train.x, train_test_dataset.train.y)
        dt_on_rf = DecisionTreeClassifier(random_state=42)
        dt_on_rf.fit(train_test_dataset.train.x, rf.predict(train_test_dataset.train.x))    

        # model
        problem = OptimalCentroidExplainer(rf, int(params.n_clf), train_test_dataset.train.x, train_test_dataset.train.y, elementwise_runner=JoblibParallelization(backend='threading'))

        res = minimize(problem,
                       NSGA2(
                           pop_size=50,
                           verbose=True,
                       ),
                       ("n_gen", 50),
                       verbose=True,
                       save_history=True,
                       seed=42)
        min_complexity_idx = np.argmin(res.F[:, 1], axis=0)
        max_acc_idx = np.argmin(res.F[:, 0], axis=0)

        model_min_complexity, ignored = problem.build_model(res.X[min_complexity_idx])
        model_max_acc, ignored = problem.build_model(res.X[max_acc_idx])

        client.log_metric(run_id, "min_complexity_test_acc", accuracy_score(train_test_dataset.test.y, model_min_complexity.predict(train_test_dataset.test.x)))
        client.log_metric(run_id, "max_acc_complexity", res.F[max_acc_idx, 1])
        client.log_metric(run_id, "max_acc_train_acc", 1 - res.F[max_acc_idx, 0])
        client.log_metric(run_id, "min_complexity_train_acc", 1 - res.F[min_complexity_idx, 0])
        client.log_metric(run_id, "min_complexity_complexity", res.F[min_complexity_idx, 1])
        client.log_metric(run_id, "max_acc_test_acc", accuracy_score(train_test_dataset.test.y, model_max_acc.predict(train_test_dataset.test.x)))
        client.log_metric(run_id, "rf_acc", accuracy_score(train_test_dataset.test.y, rf.predict(train_test_dataset.test.x)))
        client.log_metric(run_id, "dt_acc", accuracy_score(train_test_dataset.test.y, dt.predict(train_test_dataset.test.x)))
        client.log_metric(run_id, "dt_on_rf_acc", accuracy_score(train_test_dataset.test.y, dt_on_rf.predict(train_test_dataset.test.x)))
        
        terminate_run(run_id, client=client)
    except Exception as e:
        finish_run_and_print_exception(run_id, e, client = client)

In [31]:
def create_estimator_tree(centroids, rf, trees):

    space_classifier = NearestNeighbors()
    space_classifier.fit(centroids)
    wrapped_space_classifier = nn_wrapper(space_classifier)

    tree_by_centroid = {
        idx: rf.estimators_[int(tree_idx)] for idx, tree_idx in enumerate(trees)
    }


    model = SimpleCompetenceRegionEnsemble(
        wrapped_space_classifier,
        tree_by_centroid
    )

    return model, space_classifier

In [32]:

class OptimalCentroidExplainerWithTreeSelection(ElementwiseProblem):
    def __init__(self, rf, n_clusters, x_train, y_train, *args, **kwargs):
        n_dim = x_train.shape[1]

        super().__init__(
            n_var=n_clusters * n_dim + n_clusters,  # each centroid * number of features 
            n_obj=2,
            n_constr=0,
            xl=list(np.min(x_train, axis=0)) * n_clusters + n_clusters * [0],
            xu=list(np.max(x_train, axis=0)) * n_clusters + n_clusters * [len(rf.estimators_)],
            *args,
            **kwargs
        )

        self.n_clusters = n_clusters
        self.x_train = x_train
        self.y_train = y_train
        self.n_dim = n_dim
        self.rf = rf

    def build_model(self, individual):
        n_coordinates_in_individual = self.n_dim * self.n_clusters
        centroid_coordinates = individual[:n_coordinates_in_individual]
        selected_trees = individual[n_coordinates_in_individual:]
        individual_as_centroids = individual_to_centroid(centroid_coordinates, self.n_dim)

        return create_estimator_tree(individual_as_centroids, self.rf, selected_trees)


    def _evaluate(self, individual, out, *args, **kwargs):
        n_coordinates_in_individual = self.n_dim * self.n_clusters
        centroid_coordinates = individual[:n_coordinates_in_individual]
        selected_trees = individual[n_coordinates_in_individual:]

        individual_as_centroids = pipe(
            centroid_coordinates,
            lambda x: grouper(x, self.n_dim),
            list,
            np.array,
            np.nan_to_num
        )

        try:
            model, space_classifier = create_estimator_tree(individual_as_centroids, self.rf, selected_trees)
        except Exception as e:
            print(e)
            out["F"] = [1, 9999]
            return

        skf = RepeatedKFold(n_splits=3, n_repeats=3, random_state=42)
        scores = cross_validate(model, self.x_train, self.rf.predict(self.x_train), n_jobs=-1, scoring='accuracy',
                                cv=skf, fit_params={
                'competence_region_classifier': nn_wrapper(space_classifier)
            })
        acc = scores['test_score'].mean()
        mean_depth = np.average([tree.get_depth() for tree in model.clf_by_label.values()])

        print(f"Acc = {acc}, depth = {mean_depth}")

        out["F"] = [1 - acc, mean_depth]

In [33]:
def do_experiment_with_trees(run_id):
    client = MlflowClient(tracking_uri="sqlite:///experiments.db")
    params = get_run_params(run_id, client)

    from loguru import logger
    logger.info(params)

    try:
        dataset = load_embedded_dataset('keel-embedded', params.dataset_name)
        # DATA and preprocessing

        splitter = StratifiedShuffleSplit(random_state=42, n_splits=2)
        train_test_dataset = (dataset \
                              .encode_x_to_labels() \
                              .encode_y_to_numeric_labels() \
                              .train_test_split(splitter))[int(params.dataset_split)]



        dt = DecisionTreeClassifier(random_state=42)
        dt.fit(train_test_dataset.train.x, train_test_dataset.train.y)
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(train_test_dataset.train.x, train_test_dataset.train.y)
        dt_on_rf = DecisionTreeClassifier(random_state=42)
        dt_on_rf.fit(train_test_dataset.train.x, rf.predict(train_test_dataset.train.x))


        # model
        problem = OptimalCentroidExplainerWithTreeSelection(rf, int(params.n_clf), train_test_dataset.train.x, train_test_dataset.train.y)

        res = minimize(problem,
                       NSGA2(
                           pop_size=100,
                           verbose=True,
                       ),
                       ("n_gen", 100),
                       verbose=True,
                       save_history=True,
                       seed=42)
        min_complexity_idx = np.argmin(res.F[:, 1], axis=0)
        max_acc_idx = np.argmin(res.F[:, 0], axis=0)

        model_min_complexity, ignored = problem.build_model(res.X[min_complexity_idx])
        model_max_acc, ignored = problem.build_model(res.X[max_acc_idx])

        client.log_metric(run_id, "min_complexity_test_acc", accuracy_score(train_test_dataset.test.y, model_min_complexity.predict(train_test_dataset.test.x)))
        client.log_metric(run_id, "max_acc_complexity", res.F[max_acc_idx, 1])
        client.log_metric(run_id, "max_acc_train_acc", 1 - res.F[max_acc_idx, 0])
        client.log_metric(run_id, "min_complexity_train_acc", 1 - res.F[min_complexity_idx, 0])
        client.log_metric(run_id, "min_complexity_complexity", res.F[min_complexity_idx, 1])
        client.log_metric(run_id, "max_acc_test_acc", accuracy_score(train_test_dataset.test.y, model_max_acc.predict(train_test_dataset.test.x)))
        client.log_metric(run_id, "rf_acc", accuracy_score(train_test_dataset.test.y, rf.predict(train_test_dataset.test.x)))
        client.log_metric(run_id, "dt_acc", accuracy_score(train_test_dataset.test.y, dt.predict(train_test_dataset.test.x)))
        client.log_metric(run_id, "dt_on_rf_acc", accuracy_score(train_test_dataset.test.y, dt_on_rf.predict(train_test_dataset.test.x)))

        terminate_run(run_id, client=client)
    except Exception as e:
        finish_run_and_print_exception(run_id, e, client = client)